In [5]:
import tensorflow as tf
import numpy as np
from string import punctuation
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------
# LOAD DATASET
# ---------------------------
source = []
target = []

try:
    lines =  open('Tamil Dataset.txt', 'r', encoding='utf-8').read().split('\n')

    for line in lines:
        if line.strip() == "":
            continue

        # Split using TAB (English → Tamil)
        parts = line.split('\t')
        if len(parts) < 2:
            continue

        src = parts[0].lower()
        tgt = parts[1]

        # Remove punctuation
        src = src.translate(str.maketrans('', '', punctuation))
        tgt = tgt.translate(str.maketrans('', '', punctuation))

        source.append(src)
        target.append(tgt)

except UnicodeDecodeError:
    print("Encoding error in dataset")

# ---------------------------
# CREATE VOCABULARY
# ---------------------------
source_vocab = list(set(' '.join(source)))
if ' ' in source_vocab:
    source_vocab.remove(' ')
source_vocab.insert(0, ' ')

target_vocab = list(set(' '.join(target)))
if ' ' in target_vocab:
    target_vocab.remove(' ')
target_vocab.insert(0, ' ')


source_vocab_len = len(source_vocab)
target_vocab_len = len(target_vocab)

# ---------------------------
# CHARACTER MAPPING
# ---------------------------
source_char_to_int = {char: idx for idx, char in enumerate(source_vocab)}
source_int_to_char = {idx: char for idx, char in enumerate(source_vocab)}
target_char_to_int = {char: idx for idx, char in enumerate(target_vocab)}
target_int_to_char = {idx: char for idx, char in enumerate(target_vocab)}

# ---------------------------
# CONVERT TEXT → INTEGER
# ---------------------------
source = [[source_char_to_int[c] for c in s] for s in source]
target = [[target_char_to_int[c] for c in t] for t in target]

print(source)
print(target)


# ---------------------------
# PADDING
# ---------------------------
m = max(
    max(len(seq) for seq in source),
    max(len(seq) for seq in target)
)
max_sequence_length = max(len(seq) for seq in target)
source_padded = pad_sequences(source, maxlen=m, padding='post')
target_padded = pad_sequences(target, maxlen=max_sequence_length,padding='post')

# ---------------------------
# ONE-HOT ENCODING (TARGET)
# ---------------------------
target_one_hot = []
for seq in target_padded:
 encoded = to_categorical(seq, num_classes=target_vocab_len)
 target_one_hot.append(encoded)
target_one_hot = np.array(target_one_hot)

print(target_one_hot)

# ---------------------------
# MODEL
# ---------------------------
model = Sequential([
    Embedding(source_vocab_len, 64, input_length=m),
    LSTM(128, return_sequences=True),
    Dense(target_vocab_len, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'] )

print(model.summary())

model.fit(source_padded, target_one_hot, epochs=1000)

test_input = np.array([source_padded[9]])

prediction = model.predict(test_input)[0]

output = ""
for i in prediction:
    output += target_int_to_char[np.argmax(i)]

print("Predicted Output:", output.strip())

[[1, 0, 27, 25, 10, 28, 13], [8, 11, 25, 17, 0, 22, 2, 26, 18], [1, 25, 25, 0, 26, 11, 25, 24], [26, 19, 2, 0, 1, 27, 0, 19, 10], [26, 19, 2, 0, 24, 18, 2, 26, 27], [22, 10, 7, 1, 18, 1, 13, 10, 25, 4], [27, 19, 10, 0, 27, 17, 1, 25, 10, 22], [13, 11, 25, 24, 0, 13, 2, 0, 17, 10], [26, 19, 2, 0, 1, 27, 0, 27, 19, 10], [14, 2, 0, 13, 2, 0, 27, 25, 10, 10, 28], [1, 13, 0, 17, 11, 4, 0, 3, 11, 1, 18], [27, 19, 10, 0, 21, 1, 13, 0, 19, 1, 17], [27, 19, 10, 0, 19, 1, 13, 0, 19, 1, 17], [27, 19, 10, 0, 1, 27, 0, 24, 1, 18, 22], [27, 19, 10, 0, 1, 27, 0, 10, 1, 14, 19, 13], [26, 19, 10, 3, 10, 0, 11, 3, 10, 0, 26, 10], [24, 10, 10, 28, 0, 1, 18, 0, 13, 2, 12, 8, 19], [27, 10, 10, 0, 4, 2, 12, 0, 11, 14, 11, 1, 18], [14, 1, 16, 10, 0, 1, 13, 0, 13, 2, 0, 19, 10, 3], [1, 0, 11, 13, 10, 0, 13, 2, 2, 0, 17, 12, 8, 19], [1, 25, 25, 0, 27, 10, 10, 0, 13, 2, 0, 1, 13], [1, 13, 27, 0, 12, 28, 0, 13, 2, 0, 4, 2, 12], [25, 10, 11, 16, 10, 0, 1, 13, 0, 13, 2, 0, 17, 10], [25, 1, 27, 13, 10, 18, 0, 13, 2

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 229ms/step - accuracy: 0.6148 - loss: 2.9500
Epoch 2/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 0.7278 - loss: 1.4186
Epoch 3/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.7286 - loss: 1.3923
Epoch 4/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.7271 - loss: 1.2790
Epoch 5/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.7273 - loss: 1.2407
Epoch 6/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.7277 - loss: 1.2072
Epoch 7/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.7284 - loss: 1.1669
Epoch 8/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.7295 - loss: 1.1210
Epoch 9/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.7302 - loss: 1.0978
Epoch 10/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.7302 - loss: 1.0795
Epoch 11/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 127ms/step - accuracy: 0.7295 - loss: 1.0700
Epoch 12/1000
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 213ms

In [7]:
test_input = np.array([source_padded[19]])

prediction = model.predict(test_input)[0]

output = ""
for i in prediction:
    output += target_int_to_char[np.argmax(i)]

print("Predicted Output:", output.strip())

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
Predicted Output: நாக் நிறைய சாப்பிட்டேன்
